# A2A Made Simple: Two Agents Talking to Each Other 🤝

In this notebook we build **two small AI agents** and make them **talk to each other**.

| Agent | Its job |
|---|---|
| 🌤️ **Weather Agent** | Knows the weather in a few cities |
| 🧳 **Travel Agent** | Helps you pack for a trip, and **asks the Weather Agent** for help |

By the end you will see the Travel Agent send a message to the Weather Agent and use the answer.

> No hard words needed. When a new word comes up, we explain it right there.

## What is A2A?

**A2A** stands for **Agent-to-Agent**. It is a set of rules that lets two AI agents talk to each other.

Think of two people who speak different languages. If they both agree to speak English,
they can talk. A2A is that shared language for agents.

It does not matter who built the agent or which tool they used. If both follow A2A,
they can talk.

### The 3 ideas you need

1. **Agent Card** 🪪: a *business card* for an agent. It says: "My name is X, I can do Y,
   and you can reach me at this address."
2. **Message** ✉️: what one agent sends to another, like a text message.
3. **Answer** 📦: what comes back. A2A calls this an **artifact** (it just means "the result").

That's it. Let's build it.

## Step 0: Setup

We load our OpenAI key from the `.env` file and pick a model.

In [1]:
import asyncio
import os
import threading
import time
import warnings

import httpx
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

MODEL = "openai:gpt-4o-mini"

# The "address" where the Weather Agent will live on your computer
WEATHER_AGENT_URL = "http://127.0.0.1:10001"

print("Setup done ✅")

Setup done ✅


## Step 1: Build the Weather Agent 🌤️

This is a normal AI agent. It has one **tool** (a small Python function it can use)
that looks up the weather.

> We use fake weather data so the demo always works.

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool

FAKE_WEATHER = {
    "paris": "Rainy, 12°C",
    "dubai": "Sunny, 38°C",
    "london": "Cloudy, 9°C",
    "mumbai": "Hot and humid, 33°C",
    "tokyo": "Clear sky, 18°C",
}


@tool
def get_weather(city: str) -> str:
    """Get today's weather for a city."""
    return FAKE_WEATHER.get(city.lower(), "Sorry, I don't know the weather there.")


weather_agent = create_agent(
    MODEL,
    [get_weather],
    system_prompt="You are a weather agent. Use the get_weather tool and reply in one short sentence.",
)

# Quick test: talk to it directly (no A2A yet)
result = weather_agent.invoke({"messages": [("user", "What's the weather in Paris?")]})
print("Weather Agent says:", result["messages"][-1].content)

Weather Agent says: The weather in Paris is rainy with a temperature of 12°C.


It works. But right now **only this notebook** can talk to it.

Next we put it "on the phone line" so **other agents** can reach it using A2A.

## Step 2: Give the Weather Agent a Business Card 🪪

The **Agent Card** tells other agents:
- **who** this agent is (name and description)
- **what** it can do (its skills)
- **where** to reach it (its address)

To build the card we use **4 small building blocks** from the A2A library. Think of filling
in a real business card. Each block is one part of it.

| Class | What it is | Why we need it | Business-card example |
|---|---|---|---|
| **`AgentCard`** | The whole business card | Other agents read it **first** to decide if (and how) to talk to our agent | The full card you hand to someone |
| **`AgentSkill`** | One thing the agent is good at | Lets other agents know **what to ask** this agent | "Services: Weather lookup" |
| **`AgentInterface`** | The address and the "language" to use | Tells other agents **where** to send messages and **how** to format them | "Phone: +91..., I speak English" |
| **`AgentCapabilities`** | Extra features the agent supports | Tells other agents **what is possible**, for example live updates (streaming) | "Available on WhatsApp too ✔" |

So `AgentCard` is the card. `AgentSkill`, `AgentInterface` and `AgentCapabilities` are the
pieces we write **on** the card.

In our code below:
- `AgentSkill(...)` says: "I can do a **weather lookup**."
- `AgentInterface(...)` says: "Reach me at `http://127.0.0.1:10001/` and talk to me in **JSON-RPC**."
- `AgentCapabilities(streaming=False)` says: "I send the full answer at once, not word by word."
- `AgentCard(...)` puts it all together with a name, description and version.

### What is JSON-RPC? 📨

You saw `protocol_binding="JSONRPC"` in the address part. Here's what it means in two parts:

- **JSON**: a simple text format for data, made of names and values. For example:
  `{"city": "Paris", "temp": 12}`
- **RPC** (Remote Procedure Call): asking **another computer** to do a job for you, like
  calling a function that lives somewhere else.

So **JSON-RPC** = **asking another computer to do something, writing the request in JSON.**

It works like ordering food at a restaurant. You hand the waiter a note in a fixed format:
*what you want* + *the details* + *an order number*. The kitchen sends back your food with
the **same order number**, so you know which order it belongs to.

A JSON-RPC message looks like this:

```json
// Request (what the Travel Agent sends)
{
  "jsonrpc": "2.0",                 // which version of the rules we follow
  "id": "1",                        // the order number
  "method": "SendMessage",          // what we want done
  "params": { "message": { "parts": [{ "text": "What's the weather in Paris?" }] } }  // the details
}

// Response (what the Weather Agent sends back)
{
  "jsonrpc": "2.0",
  "id": "1",                        // same order number, so we know it's our answer
  "result": { "...": "Rainy, 12°C" }
}
```

**Why we need it:** both agents must agree on **one fixed format**. Otherwise each would
write messages its own way and they wouldn't understand each other. JSON-RPC is that agreed
format. You don't write these messages yourself. The A2A library builds them for you.

In [ ]:
from a2a.types import AgentCapabilities, AgentCard, AgentInterface, AgentSkill

weather_card = AgentCard(
    name="Weather Agent",
    description="Tells you today's weather in a city.",
    version="1.0.0",
    default_input_modes=["text/plain"],
    default_output_modes=["text/plain"],
    capabilities=AgentCapabilities(streaming=False),
    supported_interfaces=[
        AgentInterface(protocol_binding="JSONRPC", url=f"{WEATHER_AGENT_URL}/", protocol_version="1.0")
    ],
    skills=[
        AgentSkill(
            id="weather",
            name="Weather lookup",
            description="Gives today's weather for a city.",
            tags=["weather"],
            examples=["What's the weather in Tokyo?"],
        )
    ],
)

print("Business card ready for:", weather_card.name)

## Step 3: Teach the Weather Agent to Answer A2A Messages 📞

When a message arrives over A2A, someone has to:
1. read the message
2. give it to our Weather Agent
3. send the answer back

That "someone" is a small class. A2A calls it an **executor**. You can think of it as
the agent's **receptionist**.

> You don't need to remember every line. Just read the comments.

### The tools we import for the receptionist 🧰

Before the code, here's what each import does.

**Helpers** (from `a2a.helpers`): small functions that save us typing.

| Name | What it does | Why we need it |
|---|---|---|
| `get_message_text` | Takes the text out of an incoming message | A2A messages are packaged up. This opens the package and gives us the plain question, like `"What's the weather in Paris?"` |
| `new_task_from_user_message` | Creates a new **task** from a message | A2A tracks every request as a task, like giving each customer an **order number** |
| `new_text_message` | Builds a message from plain text | Used to write a reply message in the right A2A format |
| `new_text_part` | Builds one piece of an answer from text | An answer is made of "parts" (text, files, and so on). We put our weather answer into a text part |

**Server building blocks**: the pieces the receptionist works with.

| Name | What it is | Why we need it | Restaurant example 🍽️ |
|---|---|---|---|
| `AgentExecutor` | The **template** for a receptionist | We make our own receptionist from it, and A2A calls our `execute()` whenever a message arrives | The job description for a waiter |
| `RequestContext` | Everything about the **current request**: the message, the task, and so on | So our code can read what was asked | The customer's order slip |
| `EventQueue` | A **line of updates** going back to the sender | Every update we put here ("started", "here's the answer", "done") is sent back to the other agent | The counter where finished dishes are placed for pickup |
| `TaskUpdater` | A helper that **updates the task's status** | Makes it easy to say "answer ready" (`add_artifact`) and "done" (`complete`) | The waiter calling out "Order #12 is ready!" |

**How they work together:**

1. A message arrives. A2A gives our receptionist (`AgentExecutor`) a **`RequestContext`** (the order slip).
2. We use **`get_message_text`** to read the question.
3. We create a task with **`new_task_from_user_message`** (the order number).
4. We ask the Weather Agent for the answer.
5. We wrap the answer with **`new_text_part`**, and the **`TaskUpdater`** puts it on the **`EventQueue`**, which sends it back.

In [4]:
from a2a.helpers import get_message_text, new_task_from_user_message, new_text_message, new_text_part
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.tasks import TaskUpdater


class WeatherReceptionist(AgentExecutor):
    """Receives A2A messages and passes them to the Weather Agent."""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # 1. Read the incoming message
        question = get_message_text(context.message)
        print(f"   🌤️  Weather Agent received: '{question}'")

        # 2. A2A keeps track of each request as a "task" (like an order number)
        task = context.current_task or new_task_from_user_message(context.message)
        await event_queue.enqueue_event(task)
        updater = TaskUpdater(event_queue, task.id, task.context_id)

        # 3. Ask our Weather Agent (run it in a helper thread so the server stays responsive)
        result = await asyncio.to_thread(weather_agent.invoke, {"messages": [("user", question)]})
        answer = result["messages"][-1].content
        print(f"   🌤️  Weather Agent replies:  '{answer}'")

        # 4. Send the answer back, and mark the task as done
        await updater.add_artifact(parts=[new_text_part(text=answer, media_type="text/plain")], name="answer")
        await updater.complete()

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError("Cancel is not supported in this demo.")


print("Receptionist ready ✅")

Receptionist ready ✅


## Step 4: Open the Door (Start the Server) 🚪

Now we start a small web server so the Weather Agent can be reached at its address.

In real life the two agents would run on **different computers**. To keep things simple, both
run on your laptop. The rules are the same.

### The tools we import to open the door 🧰

Our Weather Agent now has a business card (Step 2) and a receptionist (Step 3). To let other
agents **actually reach it over the internet**, it needs a small web server.

| Name | What it is | Why we need it | Shop example 🏪 |
|---|---|---|---|
| `Starlette` | A small Python **web app** framework | It holds the list of web addresses (URLs) our agent answers on | The shop building itself |
| `uvicorn` | A **web server** that runs the web app | It listens on a port (here `10001`) and passes incoming requests to the app | Opening the shop door and keeping it open |
| `create_agent_card_routes` | Adds the **business card address**: `/.well-known/agent-card.json` | So any agent can visit that address and read our card | The sign in the shop window |
| `create_jsonrpc_routes` | Adds the **message address** (`/`), where JSON-RPC messages arrive | So other agents have a place to send their questions | The order counter |
| `DefaultRequestHandler` | The **manager** that handles each incoming message | It checks the message, creates the task, calls our receptionist, and sends the answer back | The shop manager who passes each order to the right person |
| `InMemoryTaskStore` | A **notebook in memory** where all tasks are written down | So A2A remembers each task (order number) and its status | The order book behind the counter |

**How they work together:**

1. **`uvicorn`** keeps the door open on port `10001`.
2. A request comes in and the **`Starlette`** app checks which address was used:
   - `/.well-known/agent-card.json` → show the **business card** (`create_agent_card_routes`)
   - `/` → a **message** has arrived (`create_jsonrpc_routes`)
3. Messages go to the **`DefaultRequestHandler`** (the manager). It writes the task in the
   **`InMemoryTaskStore`** (the order book) and calls our **receptionist** from Step 3.
4. The answer goes back out the same door.

> "In memory" means the tasks are forgotten when the notebook stops. That's fine for a demo.
> Real apps would save them in a database.

In [5]:
import uvicorn
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from starlette.applications import Starlette

handler = DefaultRequestHandler(
    agent_executor=WeatherReceptionist(),
    task_store=InMemoryTaskStore(),
    agent_card=weather_card,
)

app = Starlette(routes=[*create_agent_card_routes(weather_card), *create_jsonrpc_routes(handler, "/")])

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=10001, log_level="error"))
threading.Thread(target=server.run, daemon=True).start()

# Wait a moment until the server is ready
for _ in range(40):
    try:
        if httpx.get(f"{WEATHER_AGENT_URL}/.well-known/agent-card.json", timeout=2).status_code == 200:
            break
    except Exception:
        time.sleep(0.5)

print(f"🌤️ Weather Agent is online at {WEATHER_AGENT_URL}")

🌤️ Weather Agent is online at http://127.0.0.1:10001


## Step 5: Read the Business Card 🪪

Any agent can read the business card by visiting a standard address:

```
<agent address>/.well-known/agent-card.json
```

Every A2A agent puts its card at this same spot, so other agents always know where to look.

In [6]:
card = httpx.get(f"{WEATHER_AGENT_URL}/.well-known/agent-card.json").json()

print("Name:       ", card["name"])
print("What it does:", card["description"])
print("Skills:     ", [s["name"] for s in card["skills"]])

Name:        Weather Agent
What it does: Tells you today's weather in a city.
Skills:      ['Weather lookup']


## Step 6: Send the Weather Agent a Message ✉️

Let's send one message ourselves, the A2A way.

We write a small helper, `send_a2a_message(url, text)`. It does 3 things:
1. reads the agent's **business card**
2. sends our **message**
3. returns the **answer**

We will reuse this helper in the next step.

### The tools we import to send a message 🧰

Steps 2–4 built the **receiving** side (the Weather Agent). Now we need the **sending** side:
the tools to call an A2A agent.

| Name | What it is | Why we need it | Phone-call example 📱 |
|---|---|---|---|
| `A2ACardResolver` | Fetches and reads an agent's **business card** from its address | We need the card first to learn where and how to send messages | Looking up someone's contact card before you call |
| `create_client` | Builds a **client**: the object that actually sends messages | It reads the card and sets up the right connection for us | Your phone, ready to dial that contact |
| `ClientConfig` | **Settings** for the client | Here we say `streaming=False`, which means "give me the full answer at once" | Choosing "call" instead of "live video" |
| `SendMessageRequest` | The **envelope** for our message | A2A expects messages wrapped in this format before sending | Putting your letter in an envelope |
| `Role` | Says **who is speaking**: the user or the agent | Every message must say who sent it. We use `Role.ROLE_USER` because we are asking | Saying "Hi, this is the customer calling" |

**How they work together:**

1. **`A2ACardResolver`** gets the Weather Agent's business card.
2. **`create_client`** (with our **`ClientConfig`**) builds a client from that card.
3. We write our question with `new_text_message(...)`, mark it as **`Role.ROLE_USER`**, and put it in a **`SendMessageRequest`** envelope.
4. `client.send_message(...)` sends it, and we read the answer from the returned task.

In [7]:
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.types import Role, SendMessageRequest


async def send_a2a_message(agent_url: str, text: str) -> str:
    """Send a text message to an A2A agent and return its answer."""
    # 1. Read the business card
    async with httpx.AsyncClient() as http:
        card = await A2ACardResolver(httpx_client=http, base_url=agent_url).get_agent_card()

    # 2. Send the message
    client = await create_client(agent=card, client_config=ClientConfig(streaming=False))
    request = SendMessageRequest(message=new_text_message(text, role=Role.ROLE_USER))
    task = None
    async for response in client.send_message(request):
        task = response.task
    await client.close()

    # 3. Return the answer
    return task.artifacts[0].parts[0].text


answer = await send_a2a_message(WEATHER_AGENT_URL, "What's the weather in Dubai?")
print("\nAnswer we got back:", answer)

   🌤️  Weather Agent received: 'What's the weather in Dubai?'


   🌤️  Weather Agent replies:  'The weather in Dubai is sunny with a temperature of 38°C.'

Answer we got back: The weather in Dubai is sunny with a temperature of 38°C.


We just talked to an agent **using A2A**. 🎉

So far *we* sent the message. Next, **another agent** will send it.

## Step 7: Build the Travel Agent 🧳

The Travel Agent helps people pack for a trip. It **does not know the weather**.

So we give it one tool: **"ask the Weather Agent"**. Inside that tool, it uses
`send_a2a_message`, the same helper from Step 6.

The Travel Agent decides **by itself** when to ask the Weather Agent.

In [8]:
@tool
async def ask_weather_agent(question: str) -> str:
    """Ask the Weather Agent (another agent, over A2A) about the weather in a city."""
    print(f"🧳 Travel Agent  ──►  Weather Agent:  '{question}'")
    answer = await send_a2a_message(WEATHER_AGENT_URL, question)
    print(f"🧳 Travel Agent  ◄──  Weather Agent:  '{answer}'")
    return answer


travel_agent = create_agent(
    MODEL,
    [ask_weather_agent],
    system_prompt=(
        "You are a friendly travel agent. Before giving packing advice, always ask the "
        "Weather Agent about the weather in the destination city. "
        "Then give 3 short packing tips."
    ),
)

print("Travel Agent ready ✅")

Travel Agent ready ✅


## Step 8: Watch the Two Agents Talk 🤝

We ask the **Travel Agent** a question. Watch the printed lines:

- `──►` means the Travel Agent is **sending** a message to the Weather Agent
- `◄──` means the Weather Agent is **replying**

In [9]:
async def plan_trip(question: str):
    print(f"👤 You: {question}\n")
    result = await travel_agent.ainvoke({"messages": [("user", question)]})
    print(f"\n🧳 Travel Agent's final answer:\n{result['messages'][-1].content}")


await plan_trip("I'm going to Paris tomorrow. What should I pack?")

👤 You: I'm going to Paris tomorrow. What should I pack?



🧳 Travel Agent  ──►  Weather Agent:  'What is the weather in Paris tomorrow?'


   🌤️  Weather Agent received: 'What is the weather in Paris tomorrow?'


   🌤️  Weather Agent replies:  'Tomorrow's weather in Paris is rainy with a temperature of 12°C.'
🧳 Travel Agent  ◄──  Weather Agent:  'Tomorrow's weather in Paris is rainy with a temperature of 12°C.'



🧳 Travel Agent's final answer:
Given that the weather in Paris tomorrow is rainy with a temperature of 12°C, here are three packing tips:

1. **Waterproof Jacket**: Bring a lightweight, waterproof jacket or raincoat to keep you dry during the rain.
   
2. **Layers**: Pack layers, such as a warm sweater or long-sleeve shirt, to wear under your jacket, as 12°C can feel chilly, especially with rain.

3. **Comfortable Footwear**: Choose comfortable, waterproof shoes or boots, as you'll want to stay dry while exploring the city.


Let's try one more city.

In [10]:
await plan_trip("I have a trip to Dubai this weekend. Help me pack!")

👤 You: I have a trip to Dubai this weekend. Help me pack!



🧳 Travel Agent  ──►  Weather Agent:  'What is the weather forecast for Dubai this weekend?'


   🌤️  Weather Agent received: 'What is the weather forecast for Dubai this weekend?'


   🌤️  Weather Agent replies:  'The weather in Dubai this weekend is expected to be sunny with temperatures around 38°C.'
🧳 Travel Agent  ◄──  Weather Agent:  'The weather in Dubai this weekend is expected to be sunny with temperatures around 38°C.'



🧳 Travel Agent's final answer:
Here are three packing tips for your trip to Dubai this weekend:

1. **Lightweight Clothing**: Pack breathable, lightweight fabrics like cotton and linen to stay cool in the hot weather.

2. **Sun Protection**: Don't forget sunscreen, sunglasses, and a wide-brimmed hat to protect yourself from the strong sun.

3. **Comfortable Footwear**: Bring comfortable sandals or shoes for walking, as you'll likely spend time exploring the city.


## What Just Happened? 🧠

```
 👤 You
   │  "What should I pack for Paris?"
   ▼
 🧳 Travel Agent ── A2A message ──►  🌤️ Weather Agent
                 ◄── A2A answer ───     "Rainy, 12°C"
   │
   ▼
 "Pack an umbrella, a warm jacket, waterproof shoes"
```

1. You asked the **Travel Agent** for help.
2. The Travel Agent did not know the weather, so it **sent an A2A message** to the Weather Agent.
3. The Weather Agent **replied** with the weather.
4. The Travel Agent used that reply to give you packing tips.

## Summary

| Word | Simple meaning |
|---|---|
| **A2A** | A shared language so agents can talk to each other |
| **Agent Card** | An agent's business card: name, skills, address |
| **Message** | What one agent sends to another |
| **Artifact** | The answer that comes back |
| **Task** | A2A's "order number" for each request |
| **Executor** | The receptionist that passes messages to the agent |

### Why does this matter?

- Agents built by **different teams** (or different companies) can work together.
- Each agent can stay **small and focused**: one knows weather, one plans trips.
- You can **swap** an agent for a better one without changing the others.

**Want more?** Notebook 13 goes deeper: multi-turn chats, streaming, and the raw protocol messages.

## Clean Up 🧹

Turn off the Weather Agent's server.

In [11]:
server.should_exit = True
time.sleep(1)
print("🌤️ Weather Agent is offline. Notebook complete ✅")

🌤️ Weather Agent is offline. Notebook complete ✅
